In [1]:
import psutil 
import platform 
import os
env_name = os.environ.get('CONDA_DEFAULT_ENV')
print("当前 conda 环境名：", env_name)
print(platform.system()) # 操作系统名称 
print(platform.release()) # 操作系统版本 
print(platform.machine()) # 计算机架构 
print(platform.processor()) # 处理器类型 
# CPU 信息 
print(psutil.cpu_count()) # CPU 核数 
print(psutil.cpu_freq()) # CPU 频率 
# 内存信息 
print(psutil.virtual_memory()) # 内存总量、可用内存、已用内存等

当前 conda 环境名： D:\python\cjl
Windows
11
AMD64
Intel64 Family 6 Model 154 Stepping 3, GenuineIntel
16
scpufreq(current=2500.0, min=0.0, max=2500.0)
svmem(total=16858828800, available=3115823104, percent=81.5, used=13743005696, free=3115823104)


2 卷积和池化层
2.1 卷积层计算


1.
output_h = (input_h - kernel_h + 2 * padding) + 1
output_w = (input_w - kernel_w + 2 * padding)+ 1
output_c = num_kernels
则： output_h = (32 - 5 + 2×2)/2 + 1 = 16
尺寸为 16 × 16 × 16
2.
in_channels = 3
kernel_size = 5 * 5
multiplications_per_pixel = in_channels * kernel_size
则: multiplications_per_pixel = 3 * 5 * 5 = 75
单个输出通道一个像素的乘法次数为 75 次

2.2 编程题 手动实现二维最大池化层

In [ ]:
import numpy as np

def max_pool2d_manual(input, kernel_size, stride=1, padding=0):

    # 处理输入维度
    if len(input.shape) == 2:
        input = input.reshape(1, input.shape[0], input.shape[1])
        is_2d = True
    else:
        is_2d = False
    
    C, H, W = input.shape
    
    # 处理 kernel_size
    if isinstance(kernel_size, int):
        kernel_h = kernel_w = kernel_size
    else:
        kernel_h, kernel_w = kernel_size
    
    # 添加填充
    if padding > 0:
        input_padded = np.pad(input, ((0, 0), (padding, padding), (padding, padding)), 
                              mode='constant', constant_values=0)
        H_padded = H + 2 * padding
        W_padded = W + 2 * padding
    else:
        input_padded = input
        H_padded, W_padded = H, W
    
    # 计算输出尺寸
    out_h = (H_padded - kernel_h) // stride + 1
    out_w = (W_padded - kernel_w) // stride + 1
    
    # 初始化输出
    output = np.zeros((C, out_h, out_w))
    
    # 执行池化
    for c in range(C):
        for i in range(out_h):
            for j in range(out_w):
                # 计算窗口区域
                start_h = i * stride
                end_h = start_h + kernel_h
                start_w = j * stride
                end_w = start_w + kernel_w
                
                # 取窗口最大值
                window = input_padded[c, start_h:end_h, start_w:end_w]
                output[c, i, j] = np.max(window)
    
    if is_2d:
        output = output[0]
    
    return output

# 测试代码
print("测试1: ")
input_2d = np.array([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 16]
])
print("输入:")
print(input_2d)

output = max_pool2d_manual(input_2d, kernel_size=2, stride=2)
print(f"\n池化结果 (2x2, stride=2):")
print(output)

print("\n测试2: 带填充")
output_pad = max_pool2d_manual(input_2d, kernel_size=2, stride=2, padding=1)
print(f"带填充的结果:")
print(output_pad)

print("\n测试3: 3D输入")
input_3d = np.array([
    [[1,2,3,4],[5,6,7,8],[9,10,11,12],[13,14,15,16]],
    [[16,15,14,13],[12,11,10,9],[8,7,6,5],[4,3,2,1]]
])
print(f"输入形状: {input_3d.shape}")
output_3d = max_pool2d_manual(input_3d, kernel_size=2, stride=2)
print(f"输出形状: {output_3d.shape}")
print("第一个通道结果:")
print(output_3d[0])

测试1: 2D输入 (4x4矩阵)
输入:
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]
 [13 14 15 16]]

池化结果 (2x2, stride=2):
[[ 6.  8.]
 [14. 16.]]

测试2: 带填充
带填充的结果:
[[ 1.  3.  4.]
 [ 9. 11. 12.]
 [13. 15. 16.]]

测试3: 3D输入 (2x4x4)
输入形状: (2, 4, 4)
输出形状: (2, 2, 2)
第一个通道结果:
[[ 6.  8.]
 [14. 16.]]


3 LeNet, AlexNet, VGG和NiN
3.1 VGG参数量计算
1: 单个5×5卷积层参数量
  Param = kernel_h × kernel_w × in_channels × out_channels
  = 5 × 5 × C × C = 25C²
  当C=1时: 25 个参数

2: 两个串联3×3卷积层总参数量
  单层3×3参数: 3×3×C×C = 9C²
  两层总参数: 2 × 9C² = 18C²

3.2 编程题 NiN块实现

In [5]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super().__init__()
        self.block = nn.Sequential(
            # 普通卷积层
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
            # 第一个1x1卷积
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            # 第二个1x1卷积
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.block(x)

# 测试
block = NiNBlock(in_channels=3, out_channels=64, kernel_size=3, stride=1, padding=1)
print(f"NiN块结构:\n{block}")

x = torch.randn(1, 3, 32, 32)
out = block(x)
print(f"\n输入形状: {x.shape}")
print(f"输出形状: {out.shape}")

NiN块结构:
NiNBlock(
  (block): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
    (3): ReLU()
    (4): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
    (5): ReLU()
  )
)

输入形状: torch.Size([1, 3, 32, 32])
输出形状: torch.Size([1, 64, 32, 32])


4 Inception, 批量归一化和残差网络
4.1
输入样本: [2, 4, 6, 8]
缩放参数 γ = 2
平移参数 β = 1
常数 ε = 0

步骤1: 计算均值和方差
  μ = (2+4+6+8)/4 = 5.0
  σ² = [(2-5.0)² + (4-5.0)² + (6-5.0)² + (8-5.0)²]/4 = 5.0

步骤2: 标准化
  x̂_i = (x_i - μ)/√(σ²+ε)
  x̂_1 = (2 - 5.0)/√5.0 = -1.3416
  x̂_2 = (4 - 5.0)/√5.0 = -0.4472
  x̂_3 = (6 - 5.0)/√5.0 = 0.4472
  x̂_4 = (8 - 5.0)/√5.0 = 1.3416

步骤3: 缩放和平移
  y_i = γ × x̂_i + β
  y_1 = 2 × -1.3416 + 1 = -1.6833
  y_2 = 2 × -0.4472 + 1 = 0.1056
  y_3 = 2 × 0.4472 + 1 = 1.8944
  y_4 = 2 × 1.3416 + 1 = 3.6833

最终输出: [-1.68, 0.11, 1.89, 3.68]

4.2 编程题 残差块实现

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super().__init__()
        
        # 第一个卷积层
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                               stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        # 第二个卷积层
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 如果输入输出维度不一致，用1x1卷积调整
        if use_1x1conv:
            self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1,
                                      stride=stride)
        else:
            self.shortcut = nn.Identity()
        
        self.relu = nn.ReLU()
    
    def forward(self, x):
        # 残差路径
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        
        # 跳跃连接
        shortcut = self.shortcut(residual)
        
        # 相加
        out = out + shortcut
        out = self.relu(out)
        
        return out

# 测试
print("测试残差块:")
print("\n情况1: 输入输出通道相同")
block1 = ResidualBlock(in_channels=64, out_channels=64, use_1x1conv=False)
x = torch.randn(1, 64, 32, 32)
out = block1(x)
print(f"输入: {x.shape} → 输出: {out.shape}")

print("\n情况2: 输入输出通道不同 (使用1x1卷积调整)")
block2 = ResidualBlock(in_channels=32, out_channels=64, use_1x1conv=True, stride=2)
x = torch.randn(1, 32, 32, 32)
out = block2(x)
print(f"输入: {x.shape} → 输出: {out.shape}")

print("\n残差块结构:")
print(block2)

测试残差块:

情况1: 输入输出通道相同
输入: torch.Size([1, 64, 32, 32]) → 输出: torch.Size([1, 64, 32, 32])

情况2: 输入输出通道不同 (使用1x1卷积调整)
输入: torch.Size([1, 32, 32, 32]) → 输出: torch.Size([1, 64, 16, 16])

残差块结构:
ResidualBlock(
  (conv1): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (shortcut): Conv2d(32, 64, kernel_size=(1, 1), stride=(2, 2))
  (relu): ReLU()
)


5 图形增广，微调和样式迁移
5.1
1:
原因分析:
  （1）. 底层特征提取层: 学习通用的特征(边缘、纹理等)
      这些特征在不同任务间具有可迁移性
      预训练权重已经很好，不需要大幅调整
      小学习率可以保留这些通用特征

  （2）. 顶层输出层: 学习任务特定的分类决策
      目标数据集的类别分布不同
      需要从头学习或大幅调整
      大学习率可以快速适应新任务
2:问题2：
     （1） 冻结大部分底层网络：只微调最后几层
     （2） 使用极小学习率：例如设置为原始学习率的1/10或1/100
     （3） 强化正则化：
         增加Dropout概率
         增大权重衰减系数
         使用早停法
     （4） 数据增强：应用随机裁剪、翻转、旋转等增广技术扩充训练集
     （5） 减少模型容量：如果可能，使用更小的网络或减少顶层神经元数量
     （6） 不使用微调，直接作为特征提取器：完全冻结预训练模型，仅训练一个简单的分类器

  

5.2 编程题 图像增广管道

In [8]:
from torchvision import transforms
from PIL import Image
import torch

# 创建增广管道
augmentation_pipeline = transforms.Compose([
    # 1. 随机裁剪并缩放
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
    
    # 2. 随机水平翻转 (50%概率)
    transforms.RandomHorizontalFlip(p=0.5),
    
    # 3. 随机改变颜色
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    
    # 4. 转换为张量
    transforms.ToTensor()
])

# 演示
print("\n" + "="*50)
print("演示效果:")

# 创建测试图像
test_img = Image.new('RGB', (300, 300), color='red')
print(f"原始图像: {test_img.size}")

# 应用增广
augmented = augmentation_pipeline(test_img)
print(f"增广后张量形状: {augmented.shape}")
print(f"数值范围: [{augmented.min():.2f}, {augmented.max():.2f}]")

# 展示多次增广结果
print("\n多次随机增广结果:")
for i in range(3):
    aug_img = augmentation_pipeline(test_img)
    print(f"  第{i+1}次: 形状={aug_img.shape}")


演示效果:
原始图像: (300, 300)
增广后张量形状: torch.Size([3, 224, 224])
数值范围: [0.00, 0.71]

多次随机增广结果:
  第1次: 形状=torch.Size([3, 224, 224])
  第2次: 形状=torch.Size([3, 224, 224])
  第3次: 形状=torch.Size([3, 224, 224])


6 目标检测，计算计视觉训练技巧
6.1
真实框 A: [10, 10, 50, 50]
预测框 B: [30, 30, 70, 70]

计算交集区域
x1 = max(box_A[0], box_B[0])
y1 = max(box_A[1], box_B[1])
x2 = min(box_A[2], box_B[2])
y2 = min(box_A[3], box_B[3])
则：
  交集左上角: (30, 30)
  交集右下角: (50, 50)
  交集面积: (x2 - x1) * (y2 - y1) = 400（

计算各自面积
area_A = (box_A[2] - box_A[0]) * (box_A[3] - box_A[1])
area_B = (box_B[2] - box_B[0]) * (box_B[3] - box_B[1])
则：
  框A面积: 1600
  框B面积: 1600

计算并集面积
  并集面积 = 1600 + 1600 - 400 = 2800

步骤4: 计算IoU
  IoU = 交集面积 / 并集面积 = 400 / 2800 = 0.1429
  IoU = 14.29%

说明:
  当IoU > 0.5时，通常认为检测正确
  此处IoU = 14.29%，检测不正确

6.2编程题 标签平滑交叉熵

In [10]:
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, labels, epsilon=0.1, num_classes=None):
    """
    标签平滑交叉熵损失
    
    参数:
        logits: 模型输出 (batch_size, num_classes)
        labels: 真实标签 (batch_size,)
        epsilon: 平滑因子
        num_classes: 类别数
    """
    if num_classes is None:
        num_classes = logits.shape[-1]
    
    batch_size = logits.shape[0]
    
    # 创建平滑标签
    smooth_labels = torch.full((batch_size, num_classes), 
                               epsilon / (num_classes - 1))
    
    # 设置真实标签位置的值
    smooth_labels.scatter_(1, labels.unsqueeze(1), 1 - epsilon)
    
    # 计算log softmax
    log_probs = F.log_softmax(logits, dim=1)
    
    # 计算损失
    loss = -(smooth_labels * log_probs).sum(dim=1).mean()
    
    return loss

# 测试
num_classes = 3
batch_size = 4
epsilon = 0.1

# 创建模拟数据
logits = torch.randn(batch_size, num_classes)
labels = torch.tensor([0, 1, 2, 0])  # 真实标签

print(f"类别数: {num_classes}")
print(f"平滑因子 ε = {epsilon}")
print(f"真实标签: {labels.tolist()}")

# 计算普通交叉熵
ce_loss = F.cross_entropy(logits, labels)
print(f"\n普通交叉熵损失: {ce_loss:.4f}")

# 计算标签平滑交叉熵
ls_loss = label_smoothing_cross_entropy(logits, labels, epsilon, num_classes)
print(f"标签平滑交叉熵损失: {ls_loss:.4f}")

# 展示平滑标签
print("\n标签平滑后的目标概率分布:")
for i in range(batch_size):
    true_label = labels[i].item()
    smooth_target = torch.full((num_classes,), epsilon / (num_classes - 1))
    smooth_target[true_label] = 1 - epsilon
    print(f"  样本{i+1}: 真实标签={true_label}")
    print(f"          平滑标签={[f'{x:.3f}' for x in smooth_target.tolist()]}")


类别数: 3
平滑因子 ε = 0.1
真实标签: [0, 1, 2, 0]

普通交叉熵损失: 1.5821
标签平滑交叉熵损失: 1.5642

标签平滑后的目标概率分布:
  样本1: 真实标签=0
          平滑标签=['0.900', '0.050', '0.050']
  样本2: 真实标签=1
          平滑标签=['0.050', '0.900', '0.050']
  样本3: 真实标签=2
          平滑标签=['0.050', '0.050', '0.900']
  样本4: 真实标签=0
          平滑标签=['0.900', '0.050', '0.050']
